# Final Training Notebook — ScienceQA Visual MCQ

This notebook is the cleaned reproducible version of the best run from `fork-of-dl-finalproject`.

Main fixes compared with the midterm repo:
- no hardcoded Kaggle username paths
- all LoRA hyperparameters are defined once, printed, and saved
- model/data paths are auto-detected, with environment-variable overrides
- supports offline Kaggle runs by loading the base model from an attached Kaggle dataset
- saves the trained LoRA adapter, processor files, config JSON, and an output zip
- includes clean validation mode plus final train+val mode, so the report can separate ablations from final LB-guided training

For the final leaderboard model, use `TRAIN_MODE = "final_train_val"`.  
For clean validation ablations, use `TRAIN_MODE = "train_only_eval"`.


In [ ]:
# 0. Environment check only.
# Do NOT pip install inside the final offline Kaggle notebook.
# For local/Colab setup, install the pinned packages from requirements.txt before running:
#     pip install -r requirements.txt

import os
import sys
import glob
import json
import time
import math
import random
import shutil
import hashlib
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from tqdm.auto import tqdm

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 1. Reproducibility + single source of truth for hyperparameters.

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # These improve repeatability, but exact GPU determinism is still not guaranteed across hardware.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True

CONFIG = {
    "project": "DL Spring 2026 Final - ScienceQA Visual MCQ",
    "seed": 42,
    "base_model_name": "HuggingFaceTB/SmolVLM-500M-Instruct",
    "img_size": 224,
    "max_context_chars": 400,

    # Best-run training setup from fork-of-dl-finalproject.
    "train_mode": os.environ.get("TRAIN_MODE", "final_train_val"),  # "final_train_val" or "train_only_eval"
    "num_epochs": 3,
    "batch_size": 1,
    "grad_accum_steps": 8,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "log_every": 50,

    # QLoRA under 5M trainable params.
    "lora_r": 8,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    "trainable_param_cap": 5_000_000,

    # Output path.
    "output_dir": os.environ.get(
        "OUTPUT_DIR",
        "/kaggle/working/final_adapter" if Path("/kaggle/working").exists() else "outputs/final_adapter"
    ),
    "make_submission_after_training": True,
}

seed_everything(CONFIG["seed"])
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)

print(json.dumps(CONFIG, indent=2))


In [ ]:
# 2. Portable path discovery.

def _is_kaggle() -> bool:
    return Path("/kaggle/input").exists()

def _print_kaggle_inputs(max_lines: int = 80):
    root = Path("/kaggle/input")
    if not root.exists():
        return
    print("Mounted Kaggle inputs:")
    shown = 0
    for p in sorted(root.rglob("*")):
        print(" ", p)
        shown += 1
        if shown >= max_lines:
            print(f"  ... shown first {max_lines} paths only")
            break

def find_data_dir() -> Path:
    """
    Finds the folder containing train.csv/test.csv.
    Works for Kaggle competition mounts and local ./data.
    """
    env = os.environ.get("DATA_DIR")
    if env:
        p = Path(env)
        if not (p / "train.csv").exists():
            raise FileNotFoundError(f"DATA_DIR={p} does not contain train.csv")
        return p

    if _is_kaggle():
        matches = sorted(Path("/kaggle/input").rglob("train.csv"))
        # Prefer folders that also contain test.csv.
        matches = sorted(matches, key=lambda x: 0 if (x.parent / "test.csv").exists() else 1)
        if not matches:
            _print_kaggle_inputs()
            raise FileNotFoundError("Could not find train.csv under /kaggle/input. Attach the competition dataset.")
        return matches[0].parent

    p = Path("data")
    if not (p / "train.csv").exists():
        raise FileNotFoundError("Local data/train.csv not found. Put competition CSVs/images under ./data or set DATA_DIR.")
    return p

def looks_like_base_model_dir(p: Path) -> bool:
    has_config = (p / "config.json").exists()
    has_model_weights = (
        (p / "model.safetensors").exists()
        or (p / "model.safetensors.index.json").exists()
        or (p / "pytorch_model.bin").exists()
        or (p / "pytorch_model.bin.index.json").exists()
    )
    # Avoid accidentally choosing a PEFT adapter-only directory.
    is_adapter = (p / "adapter_config.json").exists()
    return has_config and has_model_weights and not is_adapter

def find_base_model_source():
    """
    Offline-safe model source discovery.

    In Kaggle offline mode, attach a Kaggle dataset containing a local snapshot of:
        HuggingFaceTB/SmolVLM-500M-Instruct

    Override with:
        BASE_MODEL_DIR=/kaggle/input/<dataset>/<snapshot-folder>
    """
    env = os.environ.get("BASE_MODEL_DIR")
    if env:
        p = Path(env)
        if not looks_like_base_model_dir(p):
            raise FileNotFoundError(f"BASE_MODEL_DIR={p} does not look like a full HF base-model snapshot.")
        return str(p), True

    if _is_kaggle():
        candidates = []
        for cfg in Path("/kaggle/input").rglob("config.json"):
            parent = cfg.parent
            if looks_like_base_model_dir(parent):
                candidates.append(parent)
        # Prefer paths with smolvlm in the name.
        candidates = sorted(candidates, key=lambda p: (0 if "smol" in str(p).lower() else 1, len(str(p))))
        if not candidates:
            _print_kaggle_inputs()
            raise FileNotFoundError(
                "Offline base model snapshot not found. Attach a Kaggle dataset containing the full "
                "HuggingFaceTB/SmolVLM-500M-Instruct files, or set BASE_MODEL_DIR."
            )
        return str(candidates[0]), True

    # Local/Colab fallback. This may download from Hugging Face unless files are cached.
    # For strict offline local runs, set BASE_MODEL_DIR.
    return CONFIG["base_model_name"], False

DATA_DIR = find_data_dir()
MODEL_SOURCE, LOCAL_FILES_ONLY = find_base_model_source()

print("DATA_DIR:", DATA_DIR)
print("MODEL_SOURCE:", MODEL_SOURCE)
print("LOCAL_FILES_ONLY:", LOCAL_FILES_ONLY)


In [ ]:
# 3. Load CSVs and normalize columns.

def parse_choices(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        return json.loads(x)
    raise TypeError(f"Unsupported choices value: {type(x)}")

def load_split(name: str, required: bool = True) -> pd.DataFrame | None:
    path = DATA_DIR / f"{name}.csv"
    if not path.exists():
        if required:
            raise FileNotFoundError(path)
        return None
    df = pd.read_csv(path)
    if "choices" in df.columns:
        df["choices"] = df["choices"].apply(parse_choices)
        df["num_choices"] = df["choices"].apply(len)
    return df

train_df = load_split("train", required=True)
val_df = load_split("val", required=False)
test_df = load_split("test", required=True)

print(f"train: {len(train_df):,}")
print(f"val:   {0 if val_df is None else len(val_df):,}")
print(f"test:  {len(test_df):,}")
print("Columns:", list(train_df.columns))

display(train_df.head(2))


In [ ]:
# 4. Prompt construction and image loading.

CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    context_parts = []

    lecture = row.get("lecture", "")
    hint = row.get("hint", "")

    if pd.notna(lecture) and str(lecture).strip():
        ctx = str(lecture).strip()[:CONFIG["max_context_chars"]]
        context_parts.append(ctx)
    if pd.notna(hint) and str(hint).strip():
        ctx = str(hint).strip()[:CONFIG["max_context_chars"]]
        context_parts.append(ctx)

    context_str = "\n".join(context_parts)
    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

def _resolve_image_path(data_dir: Path, rel_path: str) -> Path:
    rel = Path(str(rel_path))
    parts = rel.parts

    candidates = [
        data_dir / rel,
        data_dir / "images" / rel,
        data_dir / rel.name,
        data_dir / "images" / rel.name,
    ]

    if len(parts) > 1 and parts[0] == "images":
        candidates.append(data_dir / Path(*parts[1:]))
        candidates.append(data_dir / "images" / Path(*parts[1:]))

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError("Image not found. Tried:\n" + "\n".join(f"  {p}" for p in candidates))

def load_image(data_dir: Path, rel_path: str, img_size: int = None) -> Image.Image:
    if img_size is None:
        img_size = CONFIG["img_size"]
    return Image.open(_resolve_image_path(data_dir, rel_path)).convert("RGB").resize(
        (img_size, img_size),
        Image.BICUBIC,
    )

print(build_prompt(train_df.iloc[0], include_answer=True))


In [ ]:
# 5. Dataset objects.

class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, is_train: bool):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        item = {
            "image": load_image(self.data_dir, row["image_path"]),
            "text": build_prompt(row, include_answer=self.is_train),
            "choices": row["choices"],
            "answer": int(row["answer"]) if "answer" in row and pd.notna(row["answer"]) else -1,
            "id": row["id"] if "id" in row else idx,
        }
        return item

train_ds_preview = ScienceQADataset(train_df, DATA_DIR, is_train=True)
print("Dataset sanity check keys:", train_ds_preview[0].keys())
print("First image size:", train_ds_preview[0]["image"].size)


In [ ]:
# 6. Load processor from the same offline source as the base model.

from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(
    MODEL_SOURCE,
    local_files_only=LOCAL_FILES_ONLY,
)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print("Processor loaded from:", MODEL_SOURCE)
print("pad_token:", processor.tokenizer.pad_token)


In [ ]:
# 7. Inference and validation helpers.

LETTER_TOKEN_IDS = {
    letter: processor.tokenizer.encode(f" {letter}", add_special_tokens=False)[-1]
    for letter in CHOICE_LETTERS
}

def get_input_device(model):
    if hasattr(model, "hf_device_map"):
        for dev in model.hf_device_map.values():
            if dev in ("cpu", "disk"):
                continue
            if isinstance(dev, int):
                return torch.device(f"cuda:{dev}" if torch.cuda.is_available() else "cpu")
            if isinstance(dev, str) and dev.isdigit():
                return torch.device(f"cuda:{dev}" if torch.cuda.is_available() else "cpu")
            return torch.device(dev)
    return next(model.parameters()).device

def move_inputs_to_device(inputs: dict, device: torch.device) -> dict:
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in inputs.items()}

def predict_loglikelihood(model, image: Image.Image, prompt_text: str, num_choices: int) -> int:
    model.eval()
    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt")
    inputs = move_inputs_to_device(inputs, get_input_device(model))

    with torch.inference_mode():
        logits = model(**inputs).logits

    last_logits = logits[0, -1, :]
    log_probs = F.log_softmax(last_logits, dim=-1)
    scores = [
        log_probs[LETTER_TOKEN_IDS[CHOICE_LETTERS[i]]].item()
        for i in range(num_choices)
    ]
    return int(np.argmax(scores))

def evaluate_with_breakdowns(model, df: pd.DataFrame, data_dir: Path, limit: int | None = None):
    eval_df = df.copy().reset_index(drop=True)
    if limit is not None:
        eval_df = eval_df.head(limit)

    preds = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating"):
        image = load_image(data_dir, row["image_path"])
        prompt = build_prompt(row, include_answer=False)
        pred = predict_loglikelihood(model, image, prompt, int(row["num_choices"]))
        preds.append(pred)

    out = eval_df.copy()
    out["pred"] = preds
    out["correct"] = (out["pred"].astype(int) == out["answer"].astype(int)).astype(int)

    metrics = {
        "n": int(len(out)),
        "accuracy": float(out["correct"].mean()) if len(out) else None,
        "breakdowns": {},
    }

    for col in ["num_choices", "subject", "topic", "category", "grade"]:
        if col in out.columns:
            metrics["breakdowns"][col] = (
                out.groupby(col)["correct"]
                .agg(["count", "mean"])
                .reset_index()
                .rename(columns={"mean": "accuracy"})
                .to_dict(orient="records")
            )

    out_dir = Path(CONFIG["output_dir"])
    out.to_csv(out_dir / "validation_predictions.csv", index=False)
    with open(out_dir / "validation_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    print(json.dumps(metrics, indent=2)[:4000])
    return out, metrics

def generate_submission(model, df: pd.DataFrame, data_dir: Path, output_path: str | Path):
    preds = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Predicting test"):
        image = load_image(data_dir, row["image_path"])
        prompt = build_prompt(row, include_answer=False)
        pred = predict_loglikelihood(model, image, prompt, int(row["num_choices"]))
        preds.append(pred)

    submission = pd.DataFrame({"id": df["id"].values, "answer": preds})
    submission.to_csv(output_path, index=False)

    assert len(submission) == len(df), "Submission row count does not match test set."
    assert submission["answer"].notna().all(), "Submission has missing answers."
    print(f"Saved submission with {len(submission)} rows -> {output_path}")
    return submission


In [ ]:
# 8. Load base model in 4-bit NF4 and attach LoRA.

from transformers import AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_SOURCE,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    local_files_only=LOCAL_FILES_ONLY,
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_cfg = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    target_modules=CONFIG["lora_target_modules"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_cfg)

def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

trainable, total = count_trainable_params(model)
print(f"Trainable params: {trainable:,}")
print(f"Total params:     {total:,}")
print(f"Trainable %:      {100 * trainable / total:.4f}%")
assert trainable <= CONFIG["trainable_param_cap"], (
    f"Trainable params {trainable:,} exceeds cap {CONFIG['trainable_param_cap']:,}"
)

# Keep the PEFT printout in notebook logs for grading.
model.print_trainable_parameters()


In [ ]:
# 9. Build train loader.

if CONFIG["train_mode"] == "train_only_eval":
    train_for_model_df = train_df
    clean_validation_df = val_df
    if clean_validation_df is None:
        raise ValueError("TRAIN_MODE=train_only_eval requires val.csv.")
elif CONFIG["train_mode"] == "final_train_val":
    if val_df is None:
        train_for_model_df = train_df
    else:
        train_for_model_df = pd.concat([train_df, val_df], ignore_index=True)
    clean_validation_df = None
else:
    raise ValueError("train_mode must be 'train_only_eval' or 'final_train_val'")

train_ds = ScienceQADataset(train_for_model_df, DATA_DIR, is_train=True)

def collate_train(batch):
    texts = [item["text"] for item in batch]
    images = [item["image"] for item in batch]

    inputs = processor(text=texts, images=images, return_tensors="pt")
    inputs = move_inputs_to_device(inputs, get_input_device(model))

    # Loss only on final answer-token position.
    labels = inputs["input_ids"].clone()
    labels[:, :-1] = -100
    inputs["labels"] = labels
    return inputs

loader_generator = torch.Generator()
loader_generator.manual_seed(CONFIG["seed"])

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    collate_fn=collate_train,
    num_workers=0,
    pin_memory=False,
    generator=loader_generator,
)

print(f"TRAIN_MODE: {CONFIG['train_mode']}")
print(f"Training examples: {len(train_for_model_df):,}")
print(f"Batches/epoch: {len(train_loader):,}")
print(f"Effective batch size: {CONFIG['batch_size'] * CONFIG['grad_accum_steps']}")


In [ ]:
# 10. Training loop.

def train_model(model, train_loader):
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"],
    )

    use_amp = torch.cuda.is_available()
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    total_batches = CONFIG["num_epochs"] * len(train_loader)
    global_batch = 0
    train_start = time.time()

    print("=" * 70)
    print("Training config")
    print(json.dumps(CONFIG, indent=2))
    print("=" * 70)

    for epoch in range(CONFIG["num_epochs"]):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        epoch_start = time.time()

        for step, batch in enumerate(train_loader):
            global_batch += 1

            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                loss = model(**batch).loss / CONFIG["grad_accum_steps"]

            scaler.scale(loss).backward()
            total_loss += loss.item() * CONFIG["grad_accum_steps"]

            # Step on exact accumulation boundary OR last batch, so no data is silently dropped.
            should_step = (
                ((step + 1) % CONFIG["grad_accum_steps"] == 0)
                or ((step + 1) == len(train_loader))
            )
            if should_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            if (step + 1) % CONFIG["log_every"] == 0 or step == 0:
                avg_loss = total_loss / (step + 1)
                elapsed_epoch = time.time() - epoch_start
                sec_per_batch = elapsed_epoch / (step + 1)
                batches_left = total_batches - global_batch
                eta_min = batches_left * sec_per_batch / 60
                print(
                    f"[epoch {epoch+1}/{CONFIG['num_epochs']} "
                    f"batch {step+1}/{len(train_loader)}] "
                    f"loss={avg_loss:.4f} | {sec_per_batch:.2f}s/batch | ETA={eta_min:.1f}min"
                )

        epoch_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} complete | avg_loss={epoch_loss:.4f} | time={(time.time()-epoch_start)/60:.1f}min")

    print(f"Training finished in {(time.time() - train_start)/60:.1f}min")
    return model

model = train_model(model, train_loader)


In [ ]:
# 11. Save adapter, processor, config, and reproducibility manifest.

output_dir = Path(CONFIG["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)

manifest = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "config": CONFIG,
    "data_dir": str(DATA_DIR),
    "model_source": str(MODEL_SOURCE),
    "local_files_only": bool(LOCAL_FILES_ONLY),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "trainable_params": int(trainable),
    "total_params": int(total),
}
with open(output_dir / "train_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

# Hash small config files for sanity. Do not hash huge model files here.
hashes = {}
for p in output_dir.glob("*"):
    if p.is_file() and p.stat().st_size < 20_000_000:
        h = hashlib.sha256()
        h.update(p.read_bytes())
        hashes[p.name] = h.hexdigest()
with open(output_dir / "artifact_hashes_sha256.json", "w") as f:
    json.dump(hashes, f, indent=2)

print("Saved adapter bundle to:", output_dir)
print("Files:")
for p in sorted(output_dir.iterdir()):
    print(" ", p.name)


In [ ]:
# 12. Optional clean validation evaluation.
# In final_train_val mode, val was used for training, so do not report this as a clean ablation.
# For clean ablation numbers in the report, rerun with:
#     TRAIN_MODE=train_only_eval

if clean_validation_df is not None:
    val_preds, val_metrics = evaluate_with_breakdowns(model, clean_validation_df, DATA_DIR)
else:
    print("Skipping clean validation because TRAIN_MODE=final_train_val.")
    print("For report ablations, rerun this notebook with TRAIN_MODE=train_only_eval.")


In [ ]:
# 13. Generate final test submission directly from the trained adapter.

if CONFIG["make_submission_after_training"]:
    submission_path = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")
    submission = generate_submission(model, test_df, DATA_DIR, submission_path)
    display(submission.head())
else:
    print("CONFIG['make_submission_after_training'] is False; skipping submission generation.")


In [ ]:
# 14. Zip adapter bundle for uploading as a Kaggle dataset or cloud artifact.

output_dir = Path(CONFIG["output_dir"])
zip_base = output_dir.with_suffix("")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=output_dir)

print("Created:", zip_path)
print("Upload this adapter folder/zip as the final model weights artifact.")
